# Step 9: 用 `tmc-gray-recon` 做一个最小训练 smoke test

这一节会把 `fishe8` 的若干 reader sample 导出成 `tmc-gray-recon` 需要的简化格式，然后跑 1 个 epoch。

这里也不再把导出逻辑封装到 `tianmoucv.utils`，而是在 notebook 中显式写出：

- 用 `TianmoucDataReader` 取 sample
- 从 `sample['rawDiff']` 取 SD
- 用 `sample['F0_HDR']` 作为 pseudo ground truth
- 写入 `gt.exr`、`sd_high.npz`、`meta.json`

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'tianmoucv').exists() and (candidate / 'tianmoucv_example').exists():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError('找不到 tianmoucv 仓库根目录，请从仓库内打开本 Notebook。')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import torch


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import cv2
import yaml

from tianmoucv.data import TianmoucDataReader
from tianmoucv.utils import ensure_tutorial_dataset

info = ensure_tutorial_dataset()
cache_root = Path.home() / '.cache' / 'tianmoucv' / 'tutorial_data'
repo_dir = cache_root / 'repos' / 'tmc-gray-recon'
repo_dir.parent.mkdir(parents=True, exist_ok=True)

if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Tianmouc/tmc-gray-recon.git', str(repo_dir)], check=True)
else:
    print('已存在仓库:', repo_dir)

## 1. 从 `fishe8` 导出一个极小训练集

In [ ]:
dataset = TianmoucDataReader(str(info.sample_dir), N=1, print_info=False)
smoke_root = cache_root / 'gray_recon_smoke'
smoke_root.mkdir(parents=True, exist_ok=True)

num_samples = min(8, len(dataset))
aop_indices = (12, 13)

for sample_idx in range(num_samples):
    sample = dataset[sample_idx]
    sample_dir = smoke_root / f'sample_{sample_idx:04d}'
    sample_dir.mkdir(parents=True, exist_ok=True)

    raw_diff = torch.as_tensor(sample['rawDiff']).cpu().numpy()
    t0 = min(aop_indices[0], raw_diff.shape[1] - 1)
    t1 = min(aop_indices[1], raw_diff.shape[1] - 1)
    sd0 = raw_diff[1:, t0].astype(np.float32)
    sd1 = raw_diff[1:, t1].astype(np.float32)

    hdr = np.asarray(sample['F0_HDR'], dtype=np.float32)
    hdr_bgr = hdr[..., ::-1]
    cv2.imwrite(str(sample_dir / 'gt.exr'), hdr_bgr)
    np.savez_compressed(sample_dir / 'sd_high.npz', sd0=sd0, sd1=sd1)

    meta = {
        'high_is_8bit': True,
        'source_key': sample.get('meta', {}).get('key', info.sample_name),
        'reader_index': sample_idx,
        'aop_indices': [int(t0), int(t1)],
    }
    (sample_dir / 'meta.json').write_text(
        json.dumps(meta, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )

print('smoke dataset:', smoke_root)
print('目录数量:', len([p for p in smoke_root.iterdir() if p.is_dir()]))

## 2. 写入极简配置并启动 1 epoch 训练

In [ ]:
config = {
    'data_root': str(smoke_root),
    'split_file': None,
    'norm_anchor': 1.0,
    'crop_size': [320, 640],
    'embeddim': 16,
    'num_scales': 2,
    'use_cbam': False,
    'multi_channel': True,
    'enable_norm': False,
    'pretrained_ckpt': None,
    'batch_size': 1,
    'val_batch_size': 1,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'epochs': 1,
    'num_workers': 0,
    'grad_accumulation_steps': 1,
    'use_amp': False,
    'l1_weight': 1.0,
    'ssim_weight': 0.0,
    'val_interval': 1,
}

with open(repo_dir / 'config.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)

print((repo_dir / 'config.yaml').read_text())

In [ ]:
env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = ''
subprocess.run([sys.executable, 'train.py'], cwd=repo_dir, env=env, check=True)

## 说明

这个 smoke test 的目标是“把训练链路跑通”，不是获得有意义的最终精度。认真训练时应切换到真实的 TMC-DR / SD-Gray 数据格式，并仔细设置 `config.yaml`。